# Entrenamiento vs inferencia: el sobreajuste, en directo

**Facsímil 1 · Los cimientos** — capítulo 10 (entrenamiento frente a inferencia).

El error más común y más caro del *machine learning*, el que hace que demos deslumbrantes se estrellen en
producción, es confundir **memorizar** con **aprender**. Un modelo que se sabe los datos de entrenamiento
de memoria puede tener un error bajísimo... y fracasar con cualquier dato nuevo. En este cuaderno provocas
ese fenómeno a propósito, lo mides, entiendes por qué ocurre (el **compromiso sesgo-varianza**) y lo
combates con todas las herramientas del oficio: **más datos**, **regularización** (L1 y L2), **validación
cruzada** y **parada temprana**. Y lo verás en dos modelos distintos, no en uno.

### Qué vas a aprender
- La diferencia operativa entre **entrenar** e **inferir**, y por qué solo lo segundo cuenta.
- A reconocer **infraajuste** y **sobreajuste** de un vistazo, y la **curva en U** del error de prueba.
- El **compromiso sesgo-varianza** que está detrás.
- A domesticar el sobreajuste con **más datos**, **regularización** (Ridge L2 y Lasso L1), **validación
  cruzada** y **parada temprana**, con experimentos que ejecutas tú.
- Que el sobreajuste no es de polinomios: lo verás también en un **árbol de decisión**.

### Cómo se usa
Las celdas vienen **sin ejecutar**: pulsa ▶ de arriba abajo y verás nacer cada curva y cada número.

### Cuánto cuesta
Unos 18 minutos. CPU, sin claves.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
# Colab ya lo trae. En tu maquina:  pip install numpy matplotlib scikit-learn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
np.random.seed(3)

def modelo(grado, alpha=0.0, tipo="ridge"):
    reg = LinearRegression() if alpha == 0 else (Ridge(alpha=alpha) if tipo=="ridge" else Lasso(alpha=alpha, max_iter=5000))
    return make_pipeline(PolynomialFeatures(grado), StandardScaler(), reg)
print("Listo.")


## 1. Aprender no es memorizar

Un estudiante que se aprende de memoria los exámenes pasados saca un 10 en uno idéntico, pero se hunde si
cambias una pregunta: memorizó, no entendió. Con los modelos pasa igual. Por eso, para saber si un modelo
*aprendió*, no se le evalúa con los datos que ya vio (sería repetirle el examen), sino con datos **nuevos**
apartados antes de entrenar: el conjunto de **prueba** (*test*). La regla de oro: **el test no se toca
durante el entrenamiento**. Generamos datos de una curva conocida más ruido, para comparar con la verdad.


In [ ]:
def verdad(x): return np.sin(1.4*x) + 0.3*x
N = 30
x = np.sort(np.random.uniform(0,5,N)); y = verdad(x) + np.random.normal(0,0.35,N)
idx = np.random.permutation(N); tr,te = idx[:20], idx[20:]
x_tr,y_tr = x[tr],y[tr]; x_te,y_te = x[te],y[te]
print(f"{len(tr)} puntos para ENTRENAR, {len(te)} para EVALUAR (que el modelo NO vera al entrenar).")
plt.figure(figsize=(6,3))
plt.scatter(x_tr,y_tr,c="black",s=25,label="entrenamiento")
plt.scatter(x_te,y_te,c="black",marker="x",s=55,label="prueba")
plt.plot(np.linspace(0,5,200),verdad(np.linspace(0,5,200)),"--",color="#9c9c9c",label="verdad")
plt.legend(fontsize=8); plt.title("Los datos: verdad suave + ruido"); plt.tight_layout(); plt.show()


## 2. Tres modelos: poco, lo justo y demasiado

Ajustamos polinomios de grado creciente. Grado 1 (recta): demasiado rígido, **infraajuste**. Grado 15:
tanta libertad que pasa por *todos* los puntos —incluido el **ruido**—, **sobreajuste**. En medio, la
virtud.


In [ ]:
def ajusta(g, xx, alpha=0.0, tipo="ridge"):
    return modelo(g,alpha,tipo).fit(x_tr.reshape(-1,1),y_tr).predict(xx.reshape(-1,1))
malla = np.linspace(0,5,300)
fig,axes = plt.subplots(1,3,figsize=(13,3.5),sharey=True)
for ax,g,t in zip(axes,[1,4,15],["Grado 1: infraajuste","Grado 4: en su punto","Grado 15: sobreajuste"]):
    ax.scatter(x_tr,y_tr,c="black",s=28); ax.scatter(x_te,y_te,c="black",marker="x",s=50)
    ax.plot(malla,verdad(malla),"--",color="#9c9c9c"); ax.plot(malla,ajusta(g,malla),color="black")
    ax.set_title(t); ax.set_ylim(-2.5,3.5)
plt.tight_layout(); plt.show()


Mira el panel derecho: el grado 15 hace cabriolas para tocar cada punto, pero entre ellos se dispara
lejísimos de la verdad. Con un dato nuevo (las equis), fallará. El de la izquierda es tan rígido que ni
captura la curva. El del centro acierta el equilibrio.


## 3. La curva que lo cuenta todo: la U del error de prueba

Para cada grado, el error en entrenamiento solo **baja** (más complejidad, mejor memoriza); el de prueba
baja y, cuando empieza a memorizar ruido, **sube**. Esa **U** es la firma del sobreajuste.


In [ ]:
grados = range(1,16); e_tr,e_te = [],[]
for g in grados:
    m = modelo(g).fit(x_tr.reshape(-1,1),y_tr)
    e_tr.append(mean_squared_error(y_tr,m.predict(x_tr.reshape(-1,1))))
    e_te.append(mean_squared_error(y_te,m.predict(x_te.reshape(-1,1))))
mejor = int(np.argmin(e_te))+1
plt.figure(figsize=(7,3.4))
plt.plot(list(grados),e_tr,"-o",color="#9c9c9c",label="ENTRENAMIENTO",ms=4)
plt.plot(list(grados),e_te,"-o",color="black",label="PRUEBA (no visto)",ms=4)
plt.axvline(mejor,ls="--",color="black",lw=1); plt.yscale("log")
plt.xlabel("complejidad (grado)"); plt.ylabel("error (log)"); plt.legend(fontsize=8)
plt.title("El error de prueba baja y vuelve a subir"); plt.tight_layout(); plt.show()
print(f"Error de entrenamiento minimo en grado {int(np.argmin(e_tr))+1} (siempre el mas complejo).")
print(f"Mejor error de PRUEBA en grado {mejor}: ese generaliza.")


## 4. Por qué pasa: sesgo y varianza

El error se descompone en dos fuentes opuestas. **Sesgo:** error por ser demasiado simple (domina a la
izquierda; el modelo *no puede* capturar la curva: infraajuste). **Varianza:** error por ser demasiado
sensible a los datos concretos, ruido incluido (domina a la derecha: sobreajuste). Bajar uno sube el otro:
ese es el **compromiso sesgo-varianza**. El mínimo de la U es donde su suma es menor. Si solo miraras el
error de entrenamiento (que ignora la varianza), elegirías el modelo más complejo y te equivocarías.


## 5. Defensa nº1: más datos

La forma más directa de combatir el sobreajuste: **dar más datos**. Con muchos ejemplos, al modelo
complejo ya no le compensa memorizar el ruido de unos pocos. Entrenamos el grado 15 con cada vez más
puntos y miramos su error de prueba.


In [ ]:
def err15(n):
    rng = np.random.default_rng(0)
    xx = np.sort(rng.uniform(0,5,n+200)); yy = verdad(xx)+rng.normal(0,0.35,len(xx))
    m = modelo(15).fit(xx[:n].reshape(-1,1),yy[:n])
    return mean_squared_error(yy[n:],m.predict(xx[n:].reshape(-1,1)))
tam = [15,25,50,100,300]; errs = [err15(n) for n in tam]
plt.figure(figsize=(6.5,3.2)); plt.plot(tam,errs,"-o",color="black")
plt.xlabel("nº de datos de entrenamiento"); plt.ylabel("error de prueba (grado 15)")
plt.title("Mas datos curan el sobreajuste del modelo complejo"); plt.tight_layout(); plt.show()
print("El mismo grado 15, sobreajustado con pocos datos, generaliza bien con muchos.")


## 6. Defensa nº2: regularización (L2 Ridge y L1 Lasso)

Si no puedes conseguir más datos, **atas las manos al modelo**. La regularización penaliza los coeficientes
grandes, que son los que producen las contorsiones. Dos sabores:

- **Ridge (L2):** penaliza $\sum w_i^2$. Encoge todos los coeficientes hacia cero; suaviza.
- **Lasso (L1):** penaliza $\sum |w_i|$. Tiende a poner coeficientes **exactamente a cero** (selecciona
  features): da modelos *dispersos*.

Tomamos el grado 15 (que sobreajustaba) y le subimos la regularización: el sobreajuste desaparece sin
reducir el grado.


In [ ]:
fig,axes = plt.subplots(1,3,figsize=(13,3.5),sharey=True)
for ax,(alpha,tipo,nombre) in zip(axes,[(0.0,"ridge","sin regularizar"),(0.1,"ridge","Ridge α=0.1"),(0.01,"lasso","Lasso α=0.01")]):
    ax.scatter(x_tr,y_tr,c="black",s=28); ax.plot(malla,verdad(malla),"--",color="#9c9c9c")
    ax.plot(malla,ajusta(15,malla,alpha,tipo),color="black")
    et = mean_squared_error(y_te,ajusta(15,x_te,alpha,tipo))
    ax.set_title(f"grado 15, {nombre}\nerror prueba {et:.2f}"); ax.set_ylim(-2.5,3.5)
plt.tight_layout(); plt.show()
print("Sin regularizar el grado 15 sobreajusta. Con Ridge o Lasso se calma y generaliza.")


## 7. Defensa nº3: elegir bien con validación cruzada

Si elegimos la complejidad mirando *nuestro* test, acabamos sobreajustando... ¡al test! La forma honesta:
**validación cruzada** (partir el train en pliegues, entrenar en unos y validar en otro, rotar, promediar).
La dibujamos como **curva de validación**: el error de validación cruzada según el grado, con su mínimo.


In [ ]:
# Con MUCHOS datos la validacion cruzada es estable y elige un grado coherente.
rng_cv = np.random.default_rng(5)
x_cv = np.sort(rng_cv.uniform(0,5,150)); y_cv = verdad(x_cv)+rng_cv.normal(0,0.35,150)
grados2 = range(1,16)
cv = [-cross_val_score(modelo(g),x_cv.reshape(-1,1),y_cv,cv=5,scoring="neg_mean_squared_error").mean() for g in grados2]
g_opt = list(grados2)[int(np.argmin(cv))]
plt.figure(figsize=(7,3.3))
plt.plot(list(grados2),cv,"-o",color="black",ms=4); plt.axvline(g_opt,ls="--",color="black",lw=1)
plt.yscale("log"); plt.xlabel("grado"); plt.ylabel("error validacion cruzada (log)")
plt.title(f"Curva de validacion: el optimo honesto es el grado {g_opt}"); plt.tight_layout(); plt.show()
print(f"El grado con menor error de validacion cruzada es {g_opt}, elegido SIN espiar el test.")
print("(La validacion cruzada es mas robusta que un unico split: por eso se usa para ELEGIR.)")


> **Otra defensa: la parada temprana.** Cuando un modelo se entrena por iteraciones (una red neuronal,
> por ejemplo), su error de validación baja y luego vuelve a subir, justo cuando empieza a memorizar. La
> *parada temprana* corta el entrenamiento en ese mínimo. Es la misma idea de la U, pero sobre épocas en
> vez de complejidad. La verás en acción en el cuaderno de la red neuronal de este mismo facsímil.


## 8. El sobreajuste no es de polinomios: un árbol de decisión

Para convencerte de que es un fenómeno general, cambiamos de modelo. Un **árbol de decisión** muy profundo
también memoriza (cada hoja se queda con muy pocos casos). Medimos su error de prueba según la profundidad:
otra vez la U.


In [ ]:
from sklearn.tree import DecisionTreeRegressor
profs = [1,2,3,5,8,12,20]; e_tr2,e_te2 = [],[]
for p in profs:
    m = DecisionTreeRegressor(max_depth=p,random_state=0).fit(x_tr.reshape(-1,1),y_tr)
    e_tr2.append(mean_squared_error(y_tr,m.predict(x_tr.reshape(-1,1))))
    e_te2.append(mean_squared_error(y_te,m.predict(x_te.reshape(-1,1))))
plt.figure(figsize=(7,3.3))
plt.plot(profs,e_tr2,"-o",color="#9c9c9c",label="ENTRENAMIENTO",ms=4)
plt.plot(profs,e_te2,"-o",color="black",label="PRUEBA",ms=4)
plt.xlabel("profundidad del arbol"); plt.ylabel("error"); plt.legend(fontsize=8)
plt.title("Un arbol tambien sobreajusta al crecer"); plt.tight_layout(); plt.show()
print("Misma historia con otro modelo: la complejidad sin freno memoriza y generaliza peor.")


## 9. Pruébalo tú

1. **Sube el ruido** a `0.8`: ¿se nota más el sobreajuste? Más ruido = más varianza que memorizar.
2. **Compara Ridge y Lasso** con varios `alpha`: ¿cuándo Lasso pone coeficientes a cero (mira
   `.named_steps`)? Esa dispersión es útil para seleccionar features.
3. **Mueve el punto de parada temprana**: ¿qué error de prueba tendrías si pararas en la época 50? ¿Y en
   la 400?
4. **Poda el árbol** con `min_samples_leaf` en vez de `max_depth`. ¿Otra forma de regularizar?


## 10. Errores comunes

- **Elegir el modelo por el error de entrenamiento.** Siempre prefiere el más complejo: la trampa clásica.
- **Mirar el test muchas veces** para ajustar decisiones: sobreajustas al test. Para eso está la validación
  cruzada; el test se mira **una vez**, al final.
- **Pensar que regularización = peor modelo.** Bien dosificada, casi siempre mejora la generalización.
- **Creer que el sobreajuste es solo de polinomios.** Le pasa a árboles, redes y LLM. Lo viste en dos
  modelos distintos.


## 11. Qué te llevas

- **Entrenar no es memorizar.** El juez real es el error con **datos no vistos**; un error de
  entrenamiento bajísimo puede ser una trampa.
- La **curva en U** del error de prueba es el **compromiso sesgo-varianza**.
- Cuatro defensas que usarás siempre: **más datos**, **regularización** (Ridge/Lasso), **validación
  cruzada** y **parada temprana**.
- Es un fenómeno **general** (polinomios, árboles, redes, LLM): el sobreajuste es *el* problema de
  generalizar.

**Para seguir:** el cap. 7 explica *cómo* se baja el error; el facsímil 8 entero va de no engañarte con los
datos (*leakage*, *splits*); y el facsímil 7, de medir bien para decidir.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*